In [4]:
import numpy as np
import pandas as pd
import torch
import math

In [5]:
'''Part 1 Exercises (MLP)

    E01: Tune the hyperparameters of the training to beat Andrej's best validation loss of 2.2.
    E02: Initialization matters.
        What is the loss you'd get if the predicted probabilities at initialization were perfectly uniform? What loss do we actually achieve?
        Can you tune the initialization to get a starting loss that is much more similar to the uniform loss?
    E03: Read the Bengio et al. 2003 paper (link above). Implement and try any idea from the paper. Did it work?

Part 2 Exercises (Internals & BatchNorm)

    E01: I did not get around to seeing what happens when you initialize all weights and biases to zero. Try this and train the neural net.
        You might think either that 1) the network trains just fine or 2) the network doesn't train at all, but actually it is 3) the network trains but only partially, and achieves a pretty bad final performance.
        Task: Inspect the gradients and activations to figure out what is happening and why the network is only partially training, and what part is being trained exactly.
    E02: BatchNorm has the big advantage that after training, the parameters (gamma/beta) can be "folded into" the weights of the preceding Linear layers, erasing the need to forward it at test time.
        Task: Set up a small 3-layer MLP with batchnorms, train the network, then "fold" the batchnorm gamma/beta into the preceding Linear layer's W,b by creating a new W2, b2 and erasing the batch norm. 
        Verify that this gives the same forward pass during inference'''

'\n    E01: Tune the hyperparameters of the training to beat Andrej\'s best validation loss of 2.2.\n    E02: Initialization matters.\n        What is the loss you\'d get if the predicted probabilities at initialization were perfectly uniform? What loss do we actually achieve?\n        Can you tune the initialization to get a starting loss that is much more similar to the uniform loss?\n    E03: Read the Bengio et al. 2003 paper (link above). Implement and try any idea from the paper. Did it work?\n\nPart 2 Exercises (Internals & BatchNorm)\n\n    E01: I did not get around to seeing what happens when you initialize all weights and biases to zero. Try this and train the neural net.\n        You might think either that 1) the network trains just fine or 2) the network doesn\'t train at all, but actually it is 3) the network trains but only partially, and achieves a pretty bad final performance.\n        Task: Inspect the gradients and activations to figure out what is happening and why t

In [6]:
#MLP: E01
'''
I achieved a validation set loss of 0.211 & test set loss of 0.215
I used Karpathy's last version, 10-dimensional embeddings, and 200 neurons. But I changed the learning rate applied after every 10000 iterations. 
from 0.1,0.5, 0.01. and ran another set of 50000 iterations with 0.001. 
The loss after those iterations was close to 2.83. So I ran another 50000 iterations with the learning rate as 0.5 and settled on 2.27 as the loss on my traininng data set.
and 2.11 on the validation data set. 
'''

#EDIT: I tried running 400 more runs with 0.1 as the learning rate and noticed that while the loss on the training data
#decreased to 2.0026, the loss on the validation set increased to 2.3025

"\nI achieved a validation set loss of 0.211 & test set loss of 0.215\nI used Karpathy's last version, 10-dimensional embeddings, and 200 neurons. But I changed the learning rate applied after every 10000 iterations. \nfrom 0.1,0.5, 0.01. and ran another set of 50000 iterations with 0.001. \nThe loss after those iterations was close to 2.83. So I ran another 50000 iterations with the learning rate as 0.5 and settled on 2.27 as the loss on my traininng data set.\nand 2.11 on the validation data set. \n"

In [10]:
#MLP: E02

'''
while initialising the dataset, we have 27 characters (26 alphabets +'.'). 
consideriung that everyone has the same probability(1/27), therefore the neagtive loss likelihoad (cross entropy)= -log(1/27)'''
nll=-math.log(1/27)
print(nll)

#after initialising the data using the randomised weights and biases, we apply softmax to this data. 
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)
b= torch.randn(27, generator=g, requires_grad=True)
#using this the loss obtained is around 4.4
#to obtain a loss closer to the original value, we must minise the effect of the weights
#hence on multiplying all the weights with 0.01 brings the loss closer to the desired value



3.295836866004329


In [1]:
#MLP: E03

In [2]:
#PART 2: E01

#all weights and biases are initialised to zero. W=0, b=0, therefore every single neuron ouputs exactly zero. 
#now for calculating the gradients of W we know that 
#d(loss)/dw= x*(grad of prev neuron)
#d(loss)/db= 1*(grad of prev neuron)
#since the value x is the input from the previous hidden layer, the value of dloss/dw will always be 0 and hence the weights will never learn. 
#but the value of dloss/db does get updated as it has a local gradient value of 1. so the model is not entirely not learning. 
#due to presence of this bias, the model will end up learning the data through the values of the biases, but the other values will still stay zero.

In [ ]:
#PART 2: E02
#INTUITION: 
#LINEAR LAYER: y = W·x + b 
#BATCHNORM: y2 =(y - mean) / √(var + c)
#SCALE+SHIFT: OUTPUT = W1.y2 +b1
#output = W1 · (W·x + b - mean) / √(var + c) + b1

#Folding of parameters: 
#W2 = W1 · W / √(var + c)
#b2 = W1 · (b - mean) / √(var + c) + b1